## A test space for a new metric ##

When writing and testing a new metric, it can be helpful to write the Metric class in a notebook and then run it on a single point in the sky (or a single data slice, if you're working on a metric that is not spatially varying). 
This notebook is just a way to get set up to do that writing.

In [ ]:
# Some modules you're likely to want .. add whatever is needed.
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline   
# %matplotlib notebook

In [ ]:
# Import MAF
import rubin_sim.maf as maf

You need to identify the opsim output to run on, so let's do that first. <br>
It's easy to use the current baseline simulation included with $RUBIN_SIM_DATA_DIR, so let's start with that.

In [ ]:
from rubin_sim.data import get_baseline

opsim_fname = get_baseline()
print(opsim_fname)

runName = os.path.split(opsim_fname)[-1].replace(".db", "")
print(runName)

And let's set up a slicer that will give us the observations that overlap a single point on the sky.

In [ ]:
# Specify ra / dec of the point we want to work with on the sky - in degrees.
# (these can be lists, if you want to work on multiple, limited points on the sky)
test_ra = 0.0
test_dec = -20.0
test_slicer = maf.UserPointsSlicer(test_ra, test_dec)

Now set up to work with our Metric. Remember that the Metric will work on a single DataSlice at a time -- so 
*all* of (and *only*) the observation information it receives will be the visits relevant to this test_ra/test_dec location.

In [ ]:
class MyMetricInProgress(maf.BaseMetric):
    """Documentation please. Numpy style docstrings.

    This example metric just finds the time of first observation of a particular part of the sky.

    Parameters
    ----------
    specificColumns : `str`, opt
        It's nice to be flexible about what the relevant columns are called, so specify them here.
        seeingCol = FWHMeff, etc.
    kwargs : `float`, ?
        Probably there are other things you need to set?
    """

    def __init__(self, mjd_col="observationStartMJD", **kwargs):
        self.mjd_col = mjd_col
        cols = [
            self.mjd_col,
        ]  # Add any columns that your metric needs to run -- mjdCol is just an example
        super().__init__(col=cols, units="#", **kwargs)

    def run(self, data_slice, slice_point=None):
        # This is where you write what your metric does.
        # dataSlice == the numpy recarray containing the pointing information,
        # with the columns that you said you needed in 'cols'
        # slicePoint == the information about where you're evaluating this on the sky -- ra/dec,
        # and if you specified that you need a dustmap or stellar density map, etc., those values will also
        # be defined here

        # here's a super simple example .. replace with your own code to calculate your metric values
        t_min = data_slice[self.mjd_col].min()
        return t_min


# When you re-run this cell, you may get a warning that the metric name already exists - that's ok!

The typical way to use this metric with a slicer within MAF would be as follows: 

In [ ]:
# Set up the metric
mymetric = MyMetricInProgress()

In [ ]:
# Define a sqlconstraint, if we need to just use a (large) subset of the opsim visits
sqlconstraint = None  # no constraint, make all visits available

# Examples of other potentially useful sqlconstraints:
# sqlconstraint = 'filter = "r"'  # just select the visits in a particular filter
# sqlconstraint = 'note not like "%DD%"'  # don't choose any of the DD field visits
# sqlconstraint = 'night < 365'  # only use visits in the first year of the survey

In [ ]:
# We already defined the slicer - combine the metric, slicer and sqlconstraint in a MetricBundle:
bundle = maf.MetricBundle(mymetric, test_slicer, sqlconstraint, run_name=runName)

In [ ]:
# Pass the bundle (along with any other bundles to be run on this opsim) to a MetricBundleGroup in order to
# calculate the metric bundle values.
g = maf.MetricBundleGroup({"test_metric": bundle}, opsim_fname, out_dir="test", results_db=None)
# And calculate the metric
g.run_all()

And then you can look at the `bundle.metricValues` to see what your metric calculated and how well things worked.

In [ ]:
bundle.metric_values

BUT, when you're testing a new metric, you might run this many times over .. and querying the database each time is not necessary, if you are re-using the same columns from the database. If you're re-using exactly the same data (same columns, same sqlconstraint), you can skip the query.

In [ ]:
# g.simData is the simulation visit data that the previous MetricBundleGroup queried from the database
g.sim_data[0:2]

In [ ]:
# redefine your metric and rerun the cell where it was defined (above) -
#  I'll swap the minimum time to the maximum, for this example, just so you can see the result changed
# and then set up a new metric object:
mymetric = MyMetricInProgress()  # version X
# and set up a new MetricBundle object
bundle = maf.MetricBundle(mymetric, test_slicer, sqlconstraint, run_name=runName)

# Then set up a NEW and DIFFERENTLY NAMED MetricBundleGroup
g2 = maf.MetricBundleGroup({"test_metric": bundle}, opsim_fname, out_dir="test", results_db=None)

In [ ]:
# But then run it like this:
g2.run_current(sim_data=g.sim_data, constraint=sqlconstraint)

In [ ]:
# See new bundle metric values
bundle.metric_values

Once you are satisifed your metric is working as you expect, it's time to calculate it on a larger scale. 
Let's assume we now want to calculate the (max) time of observation at each point all over the sky. 

In [ ]:
# Same metric
mymetric = MyMetricInProgress()
# Same constraint
constraint = sqlconstraint

# NEW SLICER
slicer = maf.HealpixSlicer(nside=64)

# Then setup a metric bundle
allsky_bundle = maf.MetricBundle(mymetric, slicer, constraint, run_name=runName)

# To avoid overwriting the previous metric bundle group (and simdata) I'll just make a new one
g_sky = maf.MetricBundleGroup({"all sky": allsky_bundle}, opsim_fname, out_dir="test", results_db=None)

g_sky.run_all()

You'll find that this wrote your metric outputs to disk (in `outDir`) and the metric values are still available in the bundle. You can also plot this now. 

In [ ]:
allsky_bundle.metric_values

In [ ]:
allsky_bundle.plot()

The MAF tutorials in rubin_sim_notebooks/maf/tutorials have more information on running with different slicers and making prettier versions of your output plots or calculating summary statistics on the metric values. 